# Bifurcated artery, clean data

Two inflow branches merge into one outlet. Unknowns: the vessel (design field $\gamma$) and the scalings $a_1$ (upper
inlet) and $a_2$ (lower inlet) of the two inlet velocity profiles (true value 1). Data: clean sinograms at every 2nd of
180 angles (90 angles). Paper section *Joint Reconstruction of Vascular Topology and Inlet Flow Parameters*
(revised manuscript; preprint [arXiv:2606.05487](https://arxiv.org/abs/2606.05487)).

**Run All** from anywhere in the repo. Outputs go to `runs/baseline_bifurcation/` (a run refuses an existing non-empty
folder: change `out_tag` to run again). Needs `bifurcation_gt_sngrm_160_dt1280_chord4.npy` and
`bifurcation_gt_design_sq80.npy` in the repo root.


In [ ]:
out_tag        = "baseline_bifurcation"   # every output goes to runs/<out_tag>/ (refuses an existing non-empty folder)
nelx           = 80
max_iter       = 300
move_limit     = 0.05
q_alpha_start  = 75.0         # paper Sec. 4.1: q_alpha 75 -> 50 (-0.25 per iteration)
q_alpha_rate   = 0.25
q_alpha_floor  = 50.0
beta_start     = 1.0          # beta 1 -> 4 (+0.05)
beta_rate      = 0.05
beta_cap       = 4.0
q_D_start      = 5.0          # q_D 5 -> 10 (+0.05)
q_D_rate       = 0.05
q_D_cap        = 10.0
alpha_max      = 34500.0      # brinkman_bound(3.45e-2, 5e-3/sqrt(10)): x10 walls
D_min          = 5e-6
plot_interval  = 25           # design snapshot every plot_interval iterations, 0 turns them off
save_figures   = True         # reconstruction, contour and convergence figures in runs/<out_tag>/outputs/
mu             = 3.45e-2      # reconstruction viscosity
D_agent        = 5e-3         # reconstruction contrast diffusivity (x5)
inlet_shift    = 0            # inlet location error, in elements
outlet_shift   = 0            # outlet location error, in elements (reconstruction only; walls on x = 1 follow)
case           = "bifurcation"  # geometry: bifurcation, stenosis or carotid (a key of case_config.CASES)
case_name      = None           # label in the outputs; None: the case
brep_file      = None           # None: the case's files from case_config.CASES (brep, GT sinograms, GT design)
gt_file        = None
gt_design_file = None
angle_mode     = "sparse"     # sparse: every 2nd angle; sparse10: every 10th angle; ultrasparse: 10 angles drawn from seed, sorted
seed           = 1            # one seed, two independent RNG streams: the ultrasparse angle draw and the noise
noise_level    = 0.0          # relative Gaussian noise on the loss-angle data, norm of e over norm of g; 0 is clean
transport_length = 1.0        # transport in physical (mesh) units
t_end_s        = 3.0
dt_target      = 1.0 / 320.0
pulse_start    = [0.025, 1.525]
pulse_end      = [0.525, 2.025]
a_min_val      = 0.25         # inlet-scale bounds and start (both inlets)
a_max_val      = 1.75
a_start        = 0.6
j_rescale_factor = 5.0        # J_scale is reset to the current J whenever J has dropped by this factor; 0 turns it off
filter_small   = 0.055        # density-filter radius as a fraction of the domain diagonal (5.5 %)
template_file  = "experiments/transport/topopt_bifurcated_vel.ipynb"   # source notebook, hashed into config.json

In [ ]:
# snapshot of the parameter cell (incl. papermill-injected values), written to config.json later
_PARAMS = {k: v for k, v in dict(globals()).items()
           if not k.startswith("_") and k not in ("In", "Out", "exit", "quit")
           and isinstance(v, (int, float, str, list, tuple, bool, type(None)))}
import sys
import os
import pathlib
# run from the repo root (the folder that contains src/), wherever the notebook was started
_root = pathlib.Path.cwd().resolve()
while not (_root / "src" / "forward.py").is_file():
    if _root == _root.parent:
        raise FileNotFoundError(f"repo root (the folder containing src/) not found above {os.getcwd()}")
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, os.getcwd())
import functools
import jax.numpy as jnp
import jax
from jax.typing import ArrayLike
import matplotlib.pyplot as plt
import numpy as np

import src.geometry as _geom
import src.mesher as _mesher
import src.material as _mat
import src.utils as _utils
import src.bc as _bc
import src.fe_fluid as _fea
import src.fe_transient_transport as _fea_transient_transport
import src.ct_radon_transform as _radon
import src.postprocessing as _post
import src.forward as _fwd
import src.optimize as _opt


import src.solver as _solv
import src.mma as _mma
import src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

_Ext = _utils.Extent

_FluidField = _fea.FluidField
_ConcentrationField = _fea_transient_transport.ConcentrationField
_cmap = _viz.fluid_cmap

# geometry: the case's files unless given explicitly (papermill may pass other data files)
sys.path.insert(0, os.path.abspath("experiments/transport"))
import case_config as _cc                  # per-geometry settings (brep, data, design, boundary edges) and the BC check helpers
assert case in _cc.CASES, f"unknown case {case!r} (expected one of {list(_cc.CASES)})"
brep_file      = brep_file      or _cc.CASES[case]["brep"]
gt_file        = gt_file        or _cc.CASES[case]["gt"]
gt_design_file = gt_design_file or _cc.CASES[case]["design"].format(n=80)   # the 80 x 80 design (resampled to nelx where needed)
case_name      = case_name      or case
_PARAMS.update(case_name=case_name, brep_file=brep_file, gt_file=gt_file, gt_design_file=gt_design_file)   # config.json: resolved names
print(f"case {case}: brep {brep_file} | data {gt_file} | design {gt_design_file}")

RUN_DIR = pathlib.Path("runs") / out_tag
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise FileExistsError(f"{RUN_DIR} exists and is not empty; choose a new out_tag")
(RUN_DIR / "outputs").mkdir(parents=True, exist_ok=True)
print(f"outputs -> {RUN_DIR.resolve()}")


In [ ]:
# To fix the latex error.
import matplotlib
matplotlib.rcParams['text.usetex'] = False

# Measured data

The measured sinograms (`gt_file`: 121 frames x 115 detector bins x 180 angles over $[0, \pi)$) were generated with a finer
forward model than the reconstruction (160 x 160 mesh, $\Delta t = 1/1280$ s, 4x more samples per ray) to avoid an
inverse crime. The loss uses a subset of the angles (`angle_mode`; default every 2nd angle, 90 angles). With
`noise_level` > 0, Gaussian noise with $\|e\|_2 / \|g\|_2$ = `noise_level` is added at those angles; `seed` sets the
random angles (`"ultrasparse"`) and the noise.


In [ ]:
#load the sinogram ground truth data
sngrm_file_import = gt_file
sngrm_grndtruth = jnp.load(sngrm_file_import)

# ---- loss angles and noise: one seed, two independent streams (angles, noise) ----
# SeedSequence(seed).spawn(2): changing how the noise is drawn can never change the angles, and vice versa.
SNGRM_CLEAN = np.asarray(sngrm_grndtruth)            # clean GT, kept for the truth files and the figures
n_theta_all = int(SNGRM_CLEAN.shape[2])
_rng_angles, _rng_noise = [np.random.default_rng(s) for s in np.random.SeedSequence(int(seed)).spawn(2)]
N_ULTRASPARSE = 10
if angle_mode == "sparse":
    LOSS_ANGLE_IDX = np.arange(0, n_theta_all, 2)
elif angle_mode == "sparse10":
    LOSS_ANGLE_IDX = np.arange(0, n_theta_all, 10)
elif angle_mode == "ultrasparse":
    LOSS_ANGLE_IDX = np.sort(_rng_angles.choice(n_theta_all, N_ULTRASPARSE, replace=False))
else:
    raise ValueError(f"unknown angle_mode {angle_mode!r} (expected 'sparse', 'sparse10' or 'ultrasparse')")
assert LOSS_ANGLE_IDX.min() >= 0 and LOSS_ANGLE_IDX.max() < n_theta_all and np.unique(LOSS_ANGLE_IDX).size == LOSS_ANGLE_IDX.size
# Noise model of the old noise notebooks (add_relative_gaussian_noise): i.i.d. standard normal e, rescaled so that
# ||e||_2 / ||g||_2 = noise_level exactly, i.e. one global sigma for all frames, detectors and angles.
# Here g = the GT sinogram at the loss angles only (all frames and detectors); the other angles stay clean and unused.
NOISE = dict(level=float(noise_level), seed=int(seed), stream="np.random.SeedSequence(seed).spawn(2)[1]",
             model="iid N(0,1) on GT[:, :, LOSS_ANGLE_IDX], rescaled to ||e||/||g|| = noise_level (global sigma)")
if noise_level > 0:
    _g = SNGRM_CLEAN[:, :, LOSS_ANGLE_IDX]
    _e = _rng_noise.standard_normal(_g.shape)
    _e = _e / np.linalg.norm(_e.ravel()) * float(noise_level) * np.linalg.norm(_g.ravel())
    _noisy = SNGRM_CLEAN.copy(); _noisy[:, :, LOSS_ANGLE_IDX] = _g + _e
    sngrm_grndtruth = jnp.asarray(_noisy)
    NOISE.update(realised_rel=float(np.linalg.norm(_e.ravel()) / np.linalg.norm(_g.ravel())), sigma=float(np.linalg.norm(_e.ravel()) / np.sqrt(_e.size)),   # RMS of the realised noise
                 noise_sha256=__import__("hashlib").sha256(np.ascontiguousarray(_e)).hexdigest())
print(f"loss angles: mode={angle_mode} seed={seed} n={LOSS_ANGLE_IDX.size} idx={LOSS_ANGLE_IDX.tolist() if LOSS_ANGLE_IDX.size <= 20 else '[0, 2, ..., 178]'}"
      f" | noise: level={noise_level}" + (f", realised {NOISE['realised_rel']:.4f}, sigma {NOISE['sigma']:.3e}" if noise_level > 0 else ""))


# Geometry and mesh

The domain is the unit square $\Omega = [0, 1] \times [0, 1]$ cm. The brep `brep/artery_no_ext.json` gives only its boundary
and the edges that carry the inlets (on $x = 0$), the outlets (on $x = 1$) and the walls. The vessel inside $\Omega$ is
unknown: it is the design field $\gamma$ (0 fluid, 1 solid) that the reconstruction recovers.

* 80 x 80 bilinear quadrilateral (Q1) elements. Velocity $\mathbf u = (u, v)$ and pressure $p$ use the same Q1 shape
  functions (equal order), stabilised by PSPG/SUPG.
* $2 \times 2$ Gauss quadrature.
* The concentration uses a second Q1 mesh on the same grid (one dof per node).


In [ ]:
geom = _geom.BrepGeometry(brep_file=brep_file)

nelx_desired = nely_desired = nelx
gauss_order = 2

_fluid_mesh = _mesher.grid_mesh_brep(
  brep=geom,
  nelx_desired=nelx_desired,
  nely_desired=nely_desired,
  dofs_per_node=3,
  gauss_order=gauss_order,
)

density = jnp.zeros(_fluid_mesh.num_elems)
_viz.plot_grid_mesh(_fluid_mesh, density)
_viz.plot_brep(geom)

In [ ]:
mat_params = _mat.FluidMaterial(
  mass_density=1.058,
  dynamic_viscosity=mu,
)

## Fluid properties and density filter

Blood (cell above): density $\rho = 1.058$ g/cm$^3$, viscosity $\mu = 0.0345$ P (parameter `mu`).

The design is smoothed by a circular density filter of radius 5.5 % of the domain diagonal (`filter_small`) and then
projected by a Heaviside threshold at $\eta = 0.5$ with sharpness $\beta$ (continuation below).


In [ ]:
elem_diag_length = float(jnp.mean(_fluid_mesh.elem_diag_length))
radius_of = lambda f: f * nelx_desired * elem_diag_length          # physical radius = f * sqrt(2) cm on the unit square

# circular density filter, radius filter_small x the domain diagonal (5.5 %)
cutoff_distance = radius_of(filter_small)
dens_filter_small = _utils.create_density_filter(
  _fluid_mesh.elem_centers, cutoff_distance=cutoff_distance, filter_type=_utils.Filters.CIRCULAR)
dens_filter = dens_filter_small                                   # the same filter in the post-processing
print(f"filter: r = {radius_of(filter_small):.4f} cm ({filter_small * nelx_desired:.2f} elements, nnz {int(dens_filter_small.nse):,})")


# Boundary conditions

**Flow** (all Dirichlet):

| Boundary | Condition |
|---|---|
| Inlets ($x = 0$) | parabolic $u$ with peak $U_c$ (corrected so the discrete flux equals the exact one), $v = 0$; each inlet is multiplied by its unknown scaling $a_k$ |
| Outlets ($x = 1$) | $p = 0$, $v = 0$ ($u$ free) |
| All other boundary edges | no slip, $u = v = 0$ |

**Concentration**: $c = c_{in}(t)$ at the inlets (injection below); no condition elsewhere (natural boundary condition).

**Characteristic velocity**: $U_c = \mathrm{Re}\, \nu / w$ with $\mathrm{Re} = 7.68$, $\nu = \mu_{ref} / \rho = 0.0345 / 1.058$
cm$^2$/s and $w$ the inlet width ($\nu$ stays at the reference value when `mu` is changed). Brep edge numbers are those of
`case_config.CASES`:

| case | inlets | outlets | $w$ (cm) | $U_c$ (cm/s) |
|---|---|---|---|---|
| bifurcation | 2: upper (edge 3, y = 0.625-0.775) and lower (edge 1, y = 0.20-0.35), each 0.15 cm wide | 1: edge 7, y = 0.175-0.35 | 0.15 | 1.6696 |


In [ ]:
assert os.path.normpath(brep_file) == os.path.normpath(_cc.CASES[case]["brep"]), f"brep_file {brep_file} does not match case {case!r}"
char_length = _cc.char_length(geom, case)  # width of the case's reference inlet (bifurcation: top inlet, 0.15)
 
reynolds_num = 7.68
MU_REF = 3.45e-2                       # viscosity the measurements were made with
RHO_REF = 1.058
nu_ref = MU_REF / RHO_REF              # held fixed; does NOT follow `mu`
char_velocity = reynolds_num * nu_ref / char_length
 
print(f"[{case_name}] mu={mu:g} (ref {MU_REF:g})  D={D_agent:g}  "
      f"char_velocity={char_velocity:.6g} (fixed)")

In [ ]:
face_tol = _fluid_mesh.elem_size[0] * 0.49

h_elem = 1.0 / nelx


# Boundary edges of the case (brep edge numbers in case_config.CASES): inlets on x = 0, outlets on x = 1, other walls.
# The walls on x = 0 are the complement of the (possibly shifted) inlets, so every boundary face carries exactly one
# condition; inlet_shift = 0 reproduces the brep. src/bc.build_fluid_bc stops the run if a face has two conditions, a
# boundary face has none, or a boundary node has no velocity condition (outlet nodes: v and p fixed, u free).
CASE = _cc.CASES[case]
INLET_EDGES  = {name: geom.edges[i] for name, i in CASE["inlets"].items()}    # inlet-scale order
OUTLET_EDGES = {i: geom.edges[i] for i in CASE["outlets"]}
WALL_EDGES   = [geom.edges[i] for i in CASE["other_walls"]]
print(f"[{case}] brep edges: inlets {CASE['inlets']}  outlets {CASE['outlets']}  other walls {CASE['other_walls']}")

FB = _bc.build_fluid_bc(_fluid_mesh, INLET_EDGES, OUTLET_EDGES, WALL_EDGES, char_velocity, face_tol, h_elem, _FluidField,
                        inlet_shift_elems=inlet_shift, outlet_shift_elems=outlet_shift, label=case)
fluid_bc, fluid_bcs_list, inlet_edges = FB["fluid_bc"], FB["bcs_list"], FB["inlet_edges"]
inlet_faces, fluid_inlet_bcs = FB["inlet_faces"], FB["inlet_bcs"]       # {inlet name: ...}, in inlet-scale order
INLET_NAMES = list(inlet_faces)
wall_faces, outlet_faces, outlet_faces_by_edge = FB["wall_faces"], FB["outlet_faces"], FB["outlet_faces_by_edge"]
if INLET_NAMES == ["top", "btm"]:                                      # bifurcation names used by older scripts
    inlet_top_faces, inlet_btm_faces = inlet_faces["top"], inlet_faces["btm"]
    fluid_inlet_top_bc, fluid_inlet_btm_bc = fluid_inlet_bcs["top"], fluid_inlet_bcs["btm"]
_g_design = np.asarray(jnp.load(gt_design_file)).reshape(-1)
if _g_design.size == _fluid_mesh.num_elems:
    print("design vs BC:", _cc.design_bc_consistency(_fluid_mesh, _g_design, inlet_faces, wall_faces, outlet_faces))

_viz.plot_bc(fluid_bcs_list, _fluid_mesh)

# Flow solver

The steady incompressible Navier-Stokes-Brinkman equations are nonlinear through the convective term. They are solved by
a modified Newton-Raphson method (at most 10 iterations, residual reduced to $10^{-8}$; linear systems by PETSc); see
`src/solver.py`. The Brinkman term $\alpha(\tilde\gamma)\,\mathbf u$ blocks the flow in solid elements; $\alpha$ is
interpolated between $\alpha_{\min}$ and $\alpha_{\max}$ by a convex RAMP of the projected design. The viscosity is
constant (Newtonian).


In [ ]:
solver_settings = {
  "linear": {
    "solver": _solv.LinearSolvers.PETSC,
    "petsc_solver": {},
  },
  "nonlinear": {"max_iter": 10, "threshold": 1.0e-8},
}
flow_solver = _fea.FluidSolver(
  mesh=_fluid_mesh,
  bc=fluid_bc,
  material=mat_params,
  solver_settings=solver_settings,
)


# Initialize

The flow solver is set up with the mesh, material, boundary conditions and solver settings. The initial guess is zero
with the Dirichlet values imposed.


In [ ]:
init_press_vel = jnp.zeros((_fluid_mesh.num_dofs,))
init_press_vel = init_press_vel.at[fluid_bc["fixed_dofs"]].set(fluid_bc["dirichlet_values"])

In [ ]:
u_velocity = init_press_vel[1 : _fluid_mesh.num_dofs : 3]
u_vel_elem = np.mean(u_velocity[_fluid_mesh.elem_nodes], axis=1)
_viz.plot_grid_mesh(_fluid_mesh, u_vel_elem)

# Concentration transport

The contrast agent follows the transient advection-diffusion equation, driven by the flow velocity, with SUPG
stabilisation and backward Euler in time (`src/fe_transient_transport.py`). The diffusivity is interpolated from
$D_{\max} = 5 \times 10^{-3}$ cm$^2$/s (fluid, parameter `D_agent`) to $D_{\min} = 5 \times 10^{-6}$ cm$^2$/s (solid) by a
convex RAMP of $1 - \tilde\gamma$.


In [ ]:
concentration_mesh = _mesher.grid_mesh_brep(
  brep=geom,
  nelx_desired=nelx_desired,
  nely_desired=nely_desired,
  dofs_per_node=1,
  gauss_order=gauss_order,
)

concentration_transport_mat = _mat.SpeciesTransportMaterial(
  D=D_agent,  # Diffusion coefficient
)

In [ ]:
# outlet condition # wall condition are not required as Neumann BCs are natural
c_in = 1.0 # Inlet concentration value


# concentration inlets (c = c_in) on the same, possibly shifted, inlet edges as the flow (src/bc.build_conc_bc)
concentration_bc_list, concentration_bc, init_concentration = _bc.build_conc_bc(
    concentration_mesh, INLET_EDGES, _ConcentrationField, c_in=c_in, inlet_shift_elems=inlet_shift, h_elem=h_elem)

In [ ]:
init_concentration_dof = init_concentration[concentration_mesh.elem_dof_mat]
elem_init_concentration = jnp.mean(init_concentration_dof, axis=1)
conc_ext = _Ext(min=jnp.min(init_concentration), max=jnp.max(init_concentration))

In [ ]:
_, ax = plt.subplots(1, 1)
ax = _viz.plot_grid_mesh(mesh=concentration_mesh, field=elem_init_concentration, ax=ax, colorbar=True)
ax.set_title("Initial Concentration Magnitude")

fig, ax = plt.subplots(figsize=(3, 1.5))
_viz.plot_bc(concentration_bc_list, concentration_mesh, ax=ax)
ax.set_title("Concentration Boundary Conditions")
plt.show()

In [ ]:
solver_settings = {
  "linear": {
    "solver": _solv.LinearSolvers.PETSC,
    "petsc_solver": {},
  },
  "nonlinear": {"max_iter": 10, "threshold": 1.0e-4},
}

concentration_solver = _fea_transient_transport.FEA(
  mesh=concentration_mesh,
  material=None,
  bc=concentration_bc,
  solver_settings=solver_settings,
)

# Time grid

$T = 3$ s with $\Delta t = 1/320$ s (960 steps). The 121 measured frames are every 8th step (0.025 s apart).


In [ ]:
t_end = t_end_s  # seconds
N_FRAMES = sngrm_grndtruth.shape[0]
num_time_steps = int(round(t_end / dt_target))
time_step_size = (t_end/num_time_steps) * jnp.ones(num_time_steps) #in seconds
FRAME_STRIDE = num_time_steps // (N_FRAMES - 1)
assert FRAME_STRIDE * (N_FRAMES - 1) == num_time_steps, "dt must give an integer stride to the GT frames"

time_step_sizes_star = time_step_size * char_velocity / transport_length # transport in physical units

# Contrast injection

$c_{in} = 1$ at all inlets during two boluses, $t \in [0.025, 0.525] \cup [1.525, 2.025]$ s; each switch is smoothed by a
raised-cosine ramp of 0.05 s.


In [ ]:
start_times = jnp.array(pulse_start)
end_times  = jnp.array(pulse_end)

In [ ]:
eff_diffusivity_ext = _utils.Extent(min=D_min, max=concentration_transport_mat.D)
print(f"eff_diffusivity_ext: min={D_min:g}  max={concentration_transport_mat.D:g}  "
      f"ratio={concentration_transport_mat.D / D_min:.3g}")

In [ ]:
characteristic_length = transport_length * jnp.ones(concentration_mesh.num_elems) # Pe = L_T U / D, physical units

In [ ]:
min_inv_permeability = 0.0
# max_inv_permeability = alpha_max



min_inv_permeability = _mat.brinkman_bound(
  mat_params.dynamic_viscosity, 100.0 * concentration_mesh.bounding_box.lx
)
# max_inv_permeability = _mat.brinkman_bound(
#   mat_params.dynamic_viscosity, 5.0e-3 * concentration_mesh.bounding_box.lx
# )

if alpha_max is None:
    max_inv_permeability = _mat.brinkman_bound(
    MU_REF, 5.0e-3 * concentration_mesh.bounding_box.lx * (80.0 / nelx_desired))
else:
    max_inv_permeability = float(alpha_max)

init_inv_permeability = _mat.brinkman_bound(
  mat_params.dynamic_viscosity, 1.0e-1 * concentration_mesh.bounding_box.lx
)   

inv_permeability_ext = _utils.Extent(min=min_inv_permeability, max=max_inv_permeability)
print(f"inv_permeability_ext: min={min_inv_permeability:g}  max={max_inv_permeability:g}  "
      f"ratio={max_inv_permeability / min_inv_permeability:.3g}")
_h = float(jnp.mean(_fluid_mesh.elem_diag_length)) / np.sqrt(2.0)   # edge, not diagonal
_delta = float(np.sqrt(mat_params.dynamic_viscosity / max_inv_permeability))
print(f"alpha_max = {max_inv_permeability:g}   penetration depth delta = {_delta:.4g}   "
      f"h = {_h:.4g}   delta/h = {_delta / _h:.3f}")

# ---- settings check ----
Re_ = char_velocity * char_length / nu_ref
print(f"flow  : char_length={char_length:.4f}  U={char_velocity:.5f}  Re={Re_:.3f}  alpha_max={max_inv_permeability:.6g}")
print(f"trans : L_T={transport_length}  D=[{eff_diffusivity_ext.min:.1e}, {eff_diffusivity_ext.max:.1e}]  Pe_fluid={transport_length*char_velocity/eff_diffusivity_ext.max:.1f}")
print(f"time  : T={t_end}  dt={float(time_step_size[0]):.6g}  steps={num_time_steps}  stride={FRAME_STRIDE}  frames={N_FRAMES}")
print(f"data  : {gt_file}  shape={tuple(sngrm_grndtruth.shape)}  pulses {np.asarray(start_times)} / {np.asarray(end_times)}")
assert abs(Re_ - reynolds_num) < 1e-9

# Inlet scalings

The Dirichlet values of each inlet are multiplied by its own unknown scaling $a_k$, optimised together with the design
(bounds $[0.25, 1.75]$, start $0.6$, true value 1). `POS` holds the positions of each inlet's values in the fixed dofs.


In [ ]:
bc_all = flow_solver.bc
fixed_np = np.asarray(bc_all["fixed_dofs"])
# positions (in fixed_dofs) of each inlet's Dirichlet values, in inlet-scale order
POS = [jnp.asarray(np.where(np.isin(fixed_np, np.asarray(_bc.process_boundary_conditions([fluid_inlet_bcs[k]], _fluid_mesh)["fixed_dofs"])))[0])
       for k in INLET_NAMES]
if INLET_NAMES == ["top", "btm"]:
    pos_top, pos_btm = POS                                             # bifurcation names used by older scripts
dirich_vals = jnp.asarray(bc_all["dirichlet_values"])
fixed = jnp.asarray(fixed_np)

# Objective

$J(\gamma, \mathbf a)$ = mean over frames, detector bins and loss angles of (predicted - measured sinogram)$^2$.
`src/forward.py` evaluates the chain design -> filter and projection -> Brinkman flow -> transport -> chord-limited
sinograms, and the gradient by automatic differentiation.


In [ ]:
num_inlets = len(INLET_NAMES)
a_ref = np.ones(num_inlets)

# Physical admissible range (the same for every inlet)
a_min = np.full(num_inlets, a_min_val)
a_max = np.full(num_inlets, a_max_val)

# Physical initial guess
a0 = np.full(num_inlets, a_start)

x0 = a0 / a_ref
LB = a_min / a_ref
UB = a_max / a_ref

In [ ]:
# Objective: design -> Brinkman flow -> SUPG transport -> chord-limited sinograms at the loss angles -> mean-square
# misfit to the data, and its gradient (src/forward.make_objective). Same signature as before:
#   objective_function(x, flow_solver, q_alpha, q_D, beta, press_vel, filt=filt) -> (J, dJ/dx, press_vel, projected design, sinograms)
objective_function = _fwd.make_objective(
    fluid_mesh=_fluid_mesh, a_ref=a_ref, default_filter=dens_filter_small,
    inv_permeability_ext=inv_permeability_ext, dirich_vals=dirich_vals, inlet_pos=POS, fixed=fixed,
    char_velocity=char_velocity, eff_diffusivity_ext=eff_diffusivity_ext,
    concentration_solver=concentration_solver, init_concentration=init_concentration,
    time_step_sizes_star=time_step_sizes_star, characteristic_length=characteristic_length, time_step_size=time_step_size,
    start_times=start_times, end_times=end_times, conc_ext=conc_ext,
    data=sngrm_grndtruth, grid=(nely_desired, nelx_desired), loss_angle_idx=LOSS_ANGLE_IDX,
    frame_stride=FRAME_STRIDE,
)

# Optimization

MMA (move limit 0.05, `max_iter` = 300 iterations) on the element densities and the inlet scalings, from the uniform
design $\gamma = 0.5$ and no volume constraint (`src/optimize.py`). Continuation over the first 100 iterations:
$q_\alpha$ 75 -> 50, $\beta$ 1 -> 4, $q_D$ 5 -> 10. The run stops at `max_iter` or when the KKT residual or the design
step falls below `kkt_tol` / `step_tol` (1e-6: not reached in the paper runs, which use all 300 iterations).
Outputs in `runs/<out_tag>/`: `config.json`, `history.npz`, `designs.npz`, `state.npz`, `iter_log.txt`.


In [ ]:
# Optimisation settings (src/optimize): continuation of (q_alpha, beta, q_D) at the 0-based epoch e (paper Sec. 4.1),
# and the inlet scalings optimised with the design. The run stops at max_iter or at MMA convergence (kkt_tol, step_tol).
CONT = _opt.Continuation(q_alpha_start=q_alpha_start, q_alpha_rate=q_alpha_rate, q_alpha_floor=q_alpha_floor,
                         beta_start=beta_start, beta_rate=beta_rate, beta_cap=beta_cap,
                         q_D_start=q_D_start, q_D_rate=q_D_rate, q_D_cap=q_D_cap)
INLET_SCALES = _opt.InletScales(a0=a0, a_ref=a_ref, a_min=a_min, a_max=a_max)

In [ ]:
kkt_tol = 1.e-6     # stop when the KKT residual < kkt_tol or the design step (L2 norm) < step_tol;
step_tol = 1.e-6    # not reached in the paper runs (smallest 2.1e-3 and 1.3e-2), so they use all max_iter
# last epoch at which any schedule is still changing (100 for the paper's schedule)
# a schedule with rate 0 (or already at its end value) contributes no continuation iterations
cont_end_iter = CONT.end_iter()

print(f"[{out_tag}] max_iter={max_iter} move_limit={move_limit:g} plot_interval={plot_interval} "
      f"cont_end_iter={cont_end_iter}")

# ---- run-output standard: config.json (all parameters, input files and hashes, src version, environment) ----
import json as _json, hashlib as _hashlib, subprocess as _subprocess, platform as _platform, datetime as _datetime, glob as _glob
def _sha256(path):
    h = _hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""): h.update(block)
    return h.hexdigest()
def _git(*args):
    try: return _subprocess.run(["git", *args], capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as ex: return f"unavailable: {ex}"
try:
    import petsc4py as _p4p; _petsc_version = _p4p.__version__
except Exception:
    _petsc_version = None
_inputs = [gt_file, gt_design_file, brep_file]
CONFIG = dict(
    out_tag=out_tag, created=_datetime.datetime.now().isoformat(timespec="seconds"),
    parameters=_PARAMS,
    derived=dict(char_length=char_length, char_velocity=float(char_velocity), reynolds=float(reynolds_num), nu_ref=float(nu_ref),
                 alpha_min=float(min_inv_permeability), alpha_max=float(max_inv_permeability),
                 D_range=[float(eff_diffusivity_ext.min), float(eff_diffusivity_ext.max)],
                 dt=float(time_step_size[0]), num_time_steps=int(num_time_steps), frame_stride=int(FRAME_STRIDE), n_frames=int(N_FRAMES),
                 n_det=int(sngrm_grndtruth.shape[1]), n_theta=int(sngrm_grndtruth.shape[2]), loss_angle_idx=np.asarray(LOSS_ANGLE_IDX).tolist(),
                 cont_end_iter=int(cont_end_iter), filter_radius=float(radius_of(filter_small)),
                 num_elems=int(_fluid_mesh.num_elems), flow_dofs=int(_fluid_mesh.num_dofs), conc_dofs=int(concentration_mesh.num_dofs),
                 transport_solver="solve_transient_linear"),
    inputs={p: dict(sha256=_sha256(p), bytes=os.path.getsize(p)) for p in _inputs},
    template=dict(path=template_file, sha256=_sha256(template_file) if os.path.exists(template_file) else None,
                  note="sha256 of the source notebook that was executed (before papermill injected the parameters)"),
    data=dict(angle_mode=angle_mode, seed=int(seed), loss_angle_idx=np.asarray(LOSS_ANGLE_IDX).tolist(), noise=NOISE),
    src=dict(git_commit=_git("rev-parse", "HEAD"), git_status_src=_git("status", "--porcelain", "src"),
             files={p: _sha256(p) for p in sorted(_glob.glob("src/*.py"))}),
    environment=dict(python=sys.version.split()[0], jax=jax.__version__, numpy=np.__version__, petsc4py=_petsc_version,
                     host=_platform.node(), platform=_platform.platform()),
)
(RUN_DIR / "config.json").write_text(_json.dumps(CONFIG, indent=1, default=str))
print(f"config.json written ({len(CONFIG['parameters'])} parameters, {len(CONFIG['src']['files'])} src files hashed)")

mma_state, history = _opt.optimize_design(
    objective_function, flow_solver, max_iter=max_iter,
    continuation=CONT, inlets=INLET_SCALES, filt=dens_filter_small,
    run_dir=RUN_DIR, grid=(nely_desired, nelx_desired), j_rescale_factor=j_rescale_factor,
    move_limit=move_limit, kkt_tol=kkt_tol, step_tol=step_tol, plot_interval=plot_interval,
    design_cmap=_cmap, out_tag=out_tag, num_time_steps=num_time_steps,
)

In [ ]:
saved_file_name = str(RUN_DIR / f"design_n{nelx_desired}_op.npy")
np.save(saved_file_name, mma_state.x)

In [ ]:
run_path = str(RUN_DIR / "state.npz")

In [ ]:
_post.save_state(
    run_path,
    mma_state, history,
    case=case_name,                       # case="bifurcation"
    inlet_shift=int(inlet_shift),
    nelx=nelx_desired, gt_file=sngrm_file_import,
    a0=a0.tolist(), a_min=a_min.tolist(), a_max=a_max.tolist(), a_ref=a_ref.tolist(),
    move_limit=move_limit, max_iter=max_iter, kkt_tol=kkt_tol, step_tol=step_tol,
    mu=float(mat_params.dynamic_viscosity), mu_params=float(mu), rho=float(mat_params.mass_density),
    D=float(concentration_transport_mat.D), D_params=float(D_agent),
    char_velocity=float(char_velocity), char_length=float(char_length),
    num_time_steps=int(num_time_steps), face_tol=float(face_tol),
    beta_schedule=f"min({beta_cap:g}, {beta_start:g} + {beta_rate:g}*e)", 
    ramp_schedule=f"max({q_alpha_floor:g}, {q_alpha_start:g} - {q_alpha_rate:g}*e)",
    qD_schedule=f"min({q_D_cap:g}, {q_D_start:g} + {q_D_rate:g}*e)",
    alpha_max=float(max_inv_permeability), D_min=float(D_min),
    out_tag=str(out_tag), brep_file=str(brep_file), gt_design_file=str(gt_design_file),
    transport_length=float(transport_length), t_end=float(t_end), dt=float(time_step_size[0]),
    frame_stride=int(FRAME_STRIDE), pulse_start=list(pulse_start), pulse_end=list(pulse_end),
    transport_solver="solve_transient_linear",
    filter_small=float(filter_small),
    angle_mode=str(angle_mode), seed=int(seed), noise_level=float(noise_level), loss_angle_idx=np.asarray(LOSS_ANGLE_IDX).tolist(),
    j_rescale_factor=float(j_rescale_factor),
)

# Post-processing

The metrics of the paper: the Dice coefficient (DSC) and $\mathrm{nRMSE}_\gamma$ of the projected design against the true
design, the velocity error $\varepsilon_u$ over the true lumen against the flow of the true design (nominal inlets,
viscosity and $\alpha_{\max}$), and the recovered inlet scalings $\mathbf a^\star$; plus the time per iteration and the
peak memory. Writes `metrics.json`, `final_fields.npz` (design and velocity) and the figures.


In [ ]:
# ---- load everything the post-processing needs from the saved run ----
st       = _post.load_state(run_path)
meta     = st["meta"]

d_var  = np.asarray(st["x"]).reshape(-1)
a_min_ = np.asarray(meta["a_min"], float)
a_ref_ = np.asarray(meta["a_ref"], float)
a0_    = np.asarray(meta["a0"], float)

gamma_rec  = d_var[:_fluid_mesh.num_elems]
a_rec      =  d_var[_fluid_mesh.num_elems:] * a_ref_
gamma_true = _post.resample_design(jnp.load(gt_design_file), nelx_desired)
 

# ---- geometry + inlet scale ----
geo   = _post.geometry_errors(gamma_rec, gamma_true, dens_filter,
                              _utils.threshold_filter, beta=4.0, label=f"n={nelx_desired}")
A_TRUE = [1.0] * num_inlets                    # the data were generated with nominal inlets
scale = _post.inlet_scale_report(a0_, a_rec, a_true=A_TRUE, label=f"n={nelx_desired}")

# ---- flow ----
def solve(g, a):
    x  = np.vstack([np.asarray(g).reshape(-1, 1), np.asarray(a, float).reshape(-1, 1)])
    pv = jnp.zeros((flow_solver.mesh.num_dofs,))
    pv = pv.at[flow_solver.bc["fixed_dofs"]].set(flow_solver.bc["dirichlet_values"])
    out = objective_function(jnp.asarray(x), flow_solver, 50.0, 10.0, 4.0, pv)
    return out[2], out[3]

pv_r, mf_r = solve(gamma_rec,  a_rec)          # the reconstruction under the run's own model (its alpha_max, mu, inlets)

# ---- truth flow: true design, nominal inlets (no shift), nominal mu and alpha_max: the model that generated the data ----
MU_TRUTH = float(MU_REF)
ALPHA_MAX_TRUTH = float(_mat.brinkman_bound(MU_REF, 5.0e-3 / np.sqrt(10.0) * concentration_mesh.bounding_box.lx))   # 34,500
ALPHA_MIN_TRUTH = float(_mat.brinkman_bound(MU_REF, 100.0 * concentration_mesh.bounding_box.lx))
FB_TRUTH = _bc.build_fluid_bc(_fluid_mesh, INLET_EDGES, OUTLET_EDGES, WALL_EDGES, char_velocity, face_tol, h_elem, _FluidField,
                              verbose=False, label=case)     # nominal inlets and outlets (no shift)
flow_solver_truth = _fea.FluidSolver(mesh=_fluid_mesh, bc=FB_TRUTH["fluid_bc"],
                                     material=_mat.FluidMaterial(mass_density=RHO_REF, dynamic_viscosity=MU_TRUTH),
                                     solver_settings=flow_solver.solver_settings)

def solve_truth(g):
    th = _utils.threshold_filter(dens_filter_small @ jnp.asarray(np.asarray(g).reshape(-1)), beta=4.0)
    brink = _mat.compute_ramp_interpolation(prop=th, ramp_penalty=50.0, mode="convex",
                                            prop_ext=_utils.Extent(min=ALPHA_MIN_TRUTH, max=ALPHA_MAX_TRUTH))
    fx = jnp.asarray(FB_TRUTH["fluid_bc"]["fixed_dofs"]); dv = jnp.asarray(FB_TRUTH["fluid_bc"]["dirichlet_values"])
    pv = _solv.modified_newton_raphson_solve(flow_solver_truth, jnp.zeros(_fluid_mesh.num_dofs).at[fx].set(dv), brink, dv)
    return pv.at[fx].set(dv), th

pv_t, mf_t = solve_truth(gamma_true)
print(f"flow reference: nominal truth (mu {MU_TRUTH:g}, alpha_max {ALPHA_MAX_TRUTH:g}, inlets unshifted); "
      f"reconstruction under the run's model (mu {float(mat_params.dynamic_viscosity):g}, alpha_max {float(max_inv_permeability):g}, inlet shift {inlet_shift:+d})")

flow = _post.flow_errors(pv_r, pv_t, _fluid_mesh,
                         fluid_mask=np.asarray(mf_t).reshape(-1) < 0.5,
                         outlet_faces=(outlet_faces if outlet_shift == 0 else            # shifted outlet: count the truth's outflow too
                                       sorted(set(map(tuple, outlet_faces)) | set(map(tuple, FB_TRUTH["outlet_faces"])))),
                         a_rec=a_rec, a_true=A_TRUE, label=f"n={nelx_desired}")

# ---- figures ----
if save_figures:
    _post.plot_comparison(_viz, concentration_mesh, geo, label=f"n={nelx_desired}",
                          savepath=str(RUN_DIR / "outputs" / f"recon_n{nelx_desired}.png"))
    _post.plot_design(_viz, concentration_mesh, geo["mat_frac_rec"], contour=True,
                      savepath=str(RUN_DIR / "outputs" / "recon_contour.svg"))
 
    if "obj" in st["history"]:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.semilogy(st["history"].get("epoch", np.arange(len(st["history"]["obj"]))),
                    st["history"]["obj"])
        ax.set_xlabel("epoch"); ax.set_ylabel("J"); ax.set_title(f"n={nelx_desired}")
        fig.tight_layout(); fig.savefig(RUN_DIR / "outputs" / "convergence.png", dpi=200)


In [ ]:
_, ax = plt.subplots(1, 1)
ax = _viz.plot_grid_mesh(mesh=concentration_mesh, field=geo["mat_frac_rec"], ax=ax, colorbar=True)
ax.set_title(saved_file_name)

In [ ]:
fig, ax = plt.subplots(1, 1)

ax = _viz.plot_grid_contour(
    mesh=concentration_mesh,
    field=np.array(geo["mat_frac_rec"]),
    ax=ax,
    cmap=_viz.fluid_cmap,
    contour_level=0.5,
    binary_fill=True,
    colorbar=False,
)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.axis("off")

fig.savefig(RUN_DIR / "outputs" / "smoothed_mat_frac.svg", format="svg", bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(
np.round(geo["mat_frac_rec"]).reshape(nely_desired, nelx_desired),
origin="lower",
cmap=_cmap,
vmin=0.0,
vmax=1.0,
interpolation="nearest",   # no mesh gridlines
aspect="equal",
)


In [ ]:
import json, pathlib

rec_json = {
    "dice":        float(geo["dice"]),                                    # DSC
    "nrmse_gamma": float(geo["nrmse"]),                                   # nRMSE_gamma (fraction)
    "eps_u_pct":   float(flow["vel_l2"] * 100.0),                         # eps_u (%), over the true lumen
    "a_recovered": [float(v) for v in np.asarray(a_rec).reshape(-1)],     # a* (true value 1)
    "iterations":  int(len(history["obj"])),
    "params": {
        "case": case_name, "geometry_case": case, "out_tag": out_tag, "nelx": int(nelx), "max_iter": int(max_iter),
        "move_limit": float(move_limit),
        "q_alpha": [q_alpha_start, q_alpha_rate, q_alpha_floor],
        "beta":    [beta_start,    beta_rate,    beta_cap],
        "q_D":     [q_D_start,     q_D_rate,     q_D_cap],
        "alpha_max": float(max_inv_permeability), "D_min": float(D_min), "mu": float(mu), "D_agent": float(D_agent),
        "inlet_shift": int(inlet_shift), "brep_file": brep_file, "gt_file": gt_file,
        "t_end": t_end, "dt": float(time_step_size[0]), "pulses": [list(pulse_start), list(pulse_end)],
        "a_bounds": [a_min_val, a_max_val], "a_start": a_start, "filter_small": filter_small,
        "angles": {"mode": angle_mode, "seed": int(seed), "idx": np.asarray(LOSS_ANGLE_IDX).tolist()},
        "noise": NOISE,
    },
    "flow_reference": {"truth": "true design, nominal inlets", "mu_truth": MU_TRUTH, "alpha_max_truth": ALPHA_MAX_TRUTH},
}


In [ ]:
# ---- run cost (time per iteration, peak memory), final_fields.npz (design and velocity), metrics.json ----
if "cost" in history:
    rec_json["cost"] = history["cost"]
_pvr = np.asarray(pv_r)
np.savez_compressed(RUN_DIR / "final_fields.npz", node_xy=np.asarray(_fluid_mesh.nodes.coords),
                    pressure=_pvr[0::3], u=_pvr[1::3], v=_pvr[2::3],
                    raw=np.asarray(gamma_rec).reshape(-1), projected=np.asarray(mf_r, dtype=np.float32).reshape(-1),
                    filtered=np.asarray(dens_filter_small @ jnp.asarray(np.asarray(gamma_rec).reshape(-1)), dtype=np.float32),
                    inlet_scale=np.asarray(a_rec).reshape(-1), nelx=nelx_desired, nely=nely_desired)
rec_json["files"] = ["config.json", "history.npz", "designs.npz", "state.npz", "final_fields.npz", "metrics.json"]
(RUN_DIR / "metrics.json").write_text(json.dumps(rec_json, indent=2, default=float))
print(json.dumps({k: v for k, v in rec_json.items() if k != "params"}, indent=2))
